# Notebook 01: Setup & Data Collection

This notebook collects all the raw data for our forensic analysis of Trump's Truth Social posts
and their impact on oil markets during the Iran crisis.

**Data sources:**
1. Trump Truth Social posts (scraped from trumpstruth.org)
2. Brent & WTI crude oil daily prices (FRED)
3. Intraday oil data via USO ETF
4. Polymarket Iran prediction market data

In [ ]:
# Environment Setup
import subprocess, sys

# Install dependencies, if needed
deps = ['pandas', 'matplotlib', 'seaborn', 'requests', 'beautifulsoup4',
        'yfinance', 'google-genai', 'lxml', 'scipy', 'statsmodels', 'python-dotenv', 'nbformat']
for dep in deps:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', dep])

import requests
import pandas as pd
import time
import os
import json
from bs4 import BeautifulSoup
from urllib.parse import urlparse, parse_qs
from dotenv import load_dotenv

load_dotenv()

# Create output directories
os.makedirs('../data/raw', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../figures', exist_ok=True)

print('Libraries loaded and directories ready.')


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip



Libraries loaded and directories ready.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## 1. Scrape Iran-Related Trump Truth Social Posts

We scrape all posts from October 2024 through March 2026 from trumpstruth.org.
This captures the full escalation timeline from when Iran tensions began.
Posts are tagged as Iran-related using keyword matching.

In [8]:
IRAN_KEYWORDS = [
    'iran', 'iranian', 'tehran', 'khamenei', 'persian', 'strait of hormuz',
    'nuclear deal', 'sanctions', 'middle east', 'oil', 'crude', 'opec',
    'bomb', 'strike', 'military', 'war', 'troops', 'attack', 'missile',
    'navy', 'gulf', 'escalat', 'de-escalat', 'peace talk', 'negotiate',
    'surrender', 'ceasefire', 'regime', 'threat'
]

def scrape_truth_posts(start_date, end_date, per_page=100, max_pages=200):
    """
    Scrape Trump's Truth Social posts from trumpstruth.org.
    Uses cursor-based pagination to collect all posts in date range.
    """
    all_posts = []
    base_url = 'https://trumpstruth.org'
    params = {
        'sort': 'desc',
        'per_page': str(per_page),
        'start_date': start_date,
        'end_date': end_date,
        'removed': 'include'
    }
    headers = {'User-Agent': 'Mozilla/5.0 (research project)'}
    page = 0

    while page < max_pages:
        page += 1
        try:
            resp = requests.get(base_url, params=params, headers=headers, timeout=30)
            resp.raise_for_status()
        except requests.RequestException as e:
            print(f'Request failed on page {page}: {e}')
            break

        soup = BeautifulSoup(resp.text, 'html.parser')
        statuses = soup.find_all('div', class_='status')
        if not statuses:
            print(f'No posts found on page {page}. Done.')
            break

        for status in statuses:
            content_div = status.find('div', class_='status__content')
            if content_div:
                paragraphs = content_div.find_all('p')
                post_text = '\n'.join(p.get_text(strip=True) for p in paragraphs)
            else:
                post_text = ''

            meta_items = status.find_all('a', class_='status-info__meta-item')
            timestamp_str, post_url, post_id = '', '', ''
            for item in meta_items:
                href = item.get('href', '')
                if '/statuses/' in href:
                    timestamp_str = item.get_text(strip=True)
                    post_url = href
                    post_id = href.split('/statuses/')[-1].strip()

            ext_link = status.find('a', class_='status__external-link')
            original_url = ext_link.get('href', '') if ext_link else ''
            is_retruth = bool(status.find(string=lambda s: s and 'ReTruthed' in str(s)))

            all_posts.append({
                'post_id': post_id,
                'timestamp_str': timestamp_str,
                'post_text': post_text,
                'truth_url': original_url,
                'is_retruth': is_retruth
            })

        print(f'Page {page}: scraped {len(statuses)} posts (total: {len(all_posts)})')

        # Find next page cursor
        pagination = soup.find('div', class_='pagination')
        next_link = None
        if pagination:
            for a in pagination.find_all('a'):
                if 'Next' in a.get_text():
                    next_link = a.get('href')
                    break

        if not next_link:
            print('No next page. Scraping complete.')
            break

        parsed = urlparse(next_link)
        next_params = parse_qs(parsed.query)
        if 'cursor' in next_params:
            params['cursor'] = next_params['cursor'][0]
        else:
            print('No cursor in next link. Done.')
            break

        time.sleep(1.0)

    df = pd.DataFrame(all_posts)
    print(f'\nTotal posts scraped: {len(df)}')
    return df

In [ ]:
# Scrape posts from Oct 2024 through Mar 2026
# This takes ~5-10 minutes
posts_df = scrape_truth_posts(
    start_date='2024-10-01',
    end_date='2026-03-27',
    per_page=100
)

# Parse timestamps and tag Iran-related posts
posts_df['timestamp'] = pd.to_datetime(posts_df['timestamp_str'],
                                        format='%B %d, %Y, %I:%M %p', errors='coerce')
posts_df = posts_df.sort_values('timestamp').reset_index(drop=True)
posts_df['iran_related'] = posts_df['post_text'].apply(
    lambda t: any(kw in t.lower() for kw in IRAN_KEYWORDS)
)

# Save
posts_df.to_csv('../data/raw/truth_posts.csv', index=False)
print(f'\nDate range: {posts_df["timestamp"].min()} to {posts_df["timestamp"].max()}')
print(f'Iran-related: {posts_df["iran_related"].sum()} / {len(posts_df)}')
print(f'Saved to data/raw/truth_posts.csv')
posts_df.head()

Page 1: scraped 102 posts (total: 102)
Page 2: scraped 102 posts (total: 204)
Page 3: scraped 101 posts (total: 305)
Page 4: scraped 101 posts (total: 406)
Page 5: scraped 107 posts (total: 513)
Page 6: scraped 106 posts (total: 619)
Page 7: scraped 114 posts (total: 733)
Page 8: scraped 100 posts (total: 833)
Page 9: scraped 124 posts (total: 957)
Page 10: scraped 141 posts (total: 1098)
Page 11: scraped 109 posts (total: 1207)
Page 12: scraped 141 posts (total: 1348)
Page 13: scraped 111 posts (total: 1459)
Page 14: scraped 144 posts (total: 1603)
Page 15: scraped 128 posts (total: 1731)
Page 16: scraped 108 posts (total: 1839)
Page 17: scraped 101 posts (total: 1940)
Page 18: scraped 144 posts (total: 2084)
Page 19: scraped 119 posts (total: 2203)
Page 20: scraped 131 posts (total: 2334)
Page 21: scraped 144 posts (total: 2478)
Page 22: scraped 102 posts (total: 2580)
Page 23: scraped 108 posts (total: 2688)
Page 24: scraped 135 posts (total: 2823)
Page 25: scraped 142 posts (total:

,post_id,timestamp_str,post_text,truth_url,is_retruth,timestamp,iran_related
0,5437,"June 18, 2023, 9:56 AM",Believe,https://truthsocial.com/@JonVoight/11056556351...,False,2023-06-18 09:56:00,False
1,5147,"August 8, 2023, 3:28 PM",The Biden economy is not working. The American...,https://truthsocial.com/@garrettventry/1108556...,False,2023-08-08 15:28:00,False
2,4995,"September 20, 2023, 7:56 PM",#IStandWithTrump@realDonaldTrump,https://truthsocial.com/@FruitSnacks/111100179...,False,2023-09-20 19:56:00,False
3,4995,"September 20, 2023, 7:56 PM",,https://truthsocial.com/@realDonaldTrump/11192...,False,2023-09-20 19:56:00,False
4,4997,"September 25, 2023, 11:11 AM",A reminder of how criminally corrupt the FBI a...,https://truthsocial.com/@GodandCountryy/111126...,False,2023-09-25 11:11:00,False


## 2. Download Daily Oil Prices from FRED

FRED provides free CSV downloads for:
- **DCOILBRENTEU**: Brent Crude Oil (Europe, $/barrel)
- **DCOILWTICO**: WTI Crude Oil (Cushing OK, $/barrel)

In [10]:
def download_fred_series(series_id, start_date, end_date, save_path):
    """Download a FRED time series as CSV."""
    url = 'https://fred.stlouisfed.org/graph/fredgraph.csv'
    params = {'id': series_id, 'cosd': start_date, 'coed': end_date}
    resp = requests.get(url, params=params)
    resp.raise_for_status()

    with open(save_path, 'w') as f:
        f.write(resp.text)

    df = pd.read_csv(save_path)
    df.columns = ['date', 'price']
    df['date'] = pd.to_datetime(df['date'])
    df['price'] = pd.to_numeric(df['price'], errors='coerce')
    df = df.dropna(subset=['price'])

    print(f'{series_id}: {len(df)} trading days, {df["date"].min().date()} to {df["date"].max().date()}')
    print(f'  Price range: ${df["price"].min():.2f} - ${df["price"].max():.2f}')
    return df

# Download Brent crude
brent_df = download_fred_series(
    'DCOILBRENTEU', '2024-10-01', '2026-03-27',
    '../data/raw/brent_daily.csv'
)

# Download WTI crude
wti_df = download_fred_series(
    'DCOILWTICO', '2024-10-01', '2026-03-27',
    '../data/raw/wti_daily.csv'
)

DCOILBRENTEU: 374 trading days, 2024-10-01 to 2026-03-23
  Price range: $59.93 - $118.42
DCOILWTICO: 365 trading days, 2024-10-01 to 2026-03-23
  Price range: $55.44 - $98.71


## 3. Download Intraday Oil Data

Intraday data proves **causation**: did the post come BEFORE the price move?

We use **USO ETF** (United States Oil Fund) via yfinance for 5-minute bars covering the last 60 days.
USO closely tracks WTI crude oil futures (~0.98 correlation) and covers all key March 2026 crisis dates:
- **Feb 28**: Iran strikes begin ($1.2M Polymarket profits from 6 new wallets)
- **Mar 9**: $38 intraday crude swing (largest in history)
- **Mar 23**: "Productive conversations" fabrication (Iran denied)

In [13]:
import yfinance as yf

# USO ETF 5-minute bars (last 60 days covers March 2026 crisis)
uso_intraday = yf.download('USO', period='60d', interval='5m')
uso_intraday = uso_intraday.reset_index()
if isinstance(uso_intraday.columns, pd.MultiIndex):
    uso_intraday.columns = [col[0] if col[1] == '' or col[1] == 'USO'
                            else f'{col[0]}_{col[1]}' for col in uso_intraday.columns]
uso_intraday = uso_intraday.rename(columns={'Datetime': 'timestamp', 'Close': 'uso_close'})
uso_intraday.to_csv('../data/raw/uso_intraday.csv', index=False)

print(f'USO intraday: {len(uso_intraday)} bars')
print(f'  Date range: {uso_intraday["timestamp"].min()} to {uso_intraday["timestamp"].max()}')
print(f'  Covers key dates: Feb 28, Mar 9, Mar 23')

[*********************100%***********************]  1 of 1 completed

USO intraday: 4557 bars
  Date range: 2025-12-31 14:30:00+00:00 to 2026-03-27 19:55:00+00:00
  Covers key dates: Feb 28, Mar 9, Mar 23


## 4. Pull Polymarket Iran Market Activity

Polymarket is a blockchain-based prediction market. We collect Iran-related markets
to analyze insider betting patterns around Trump's announcements.

Key evidence: A trader with a 93%+ win rate was identified betting on Iran military operations,
with 6 new wallets created before the February 28 strikes netting $1.2M in profits.

In [ ]:
IRAN_KW = ['iran', 'iranian', 'tehran', 'khamenei', 'strait of hormuz', 'persian gulf']

# Known high-value Iran market slugs (identified via prior CLOB API scan)
known_slugs = [
    'us-forces-in-iran-in-2025',
    'will-the-us-invade-iran-in-2025',
    'iran-x-usa-conflict-ends-before-july',
    'us-military-action-against-iran-before-may',
    'will-the-iranian-regime-survive-us-military-strikes',
    'will-the-iranian-regime-fall-in-2025',
    'will-the-us-officially-declare-war-on-iran-in-2025',
    'iran-gets-a-nuke-despite-us-strikes',
    'iranian-ship-zagros-confirmed-sunk-by-march-18',
    'another-us-military-action-against-iran-before-2026',
    'israel-strikes-iran-before-2026',
    'congress-authorizes-military-force-against-iran-in-2025',
    'will-iran-close-the-strait-of-hormuz-in-2025',
    'iran-nuke-in-2025', 'us-x-iran-nuclear-deal-in-2025',
    'will-trump-meet-with-ali-khamenei-in-2025',
    'will-iran-withdraw-from-the-npt-in-2025',
    'israel-x-iran-ceasefire-broken-by-december-31-374',
    'trump-x-khamenei-talk-in-2025',
    'israel-military-action-against-iran-before-2026',
    'iran-military-action-against-israel-before-july',
    'will-israel-attack-iran-before-may',
    'will-iran-attack-israel',
    'iran-strike-on-israel-before-november',
    'israel-military-response-against-iran-in-october',
    'will-israel-target-an-iranian-nuclear-facility',
    'will-israel-target-an-iranian-oil-or-gas-facility',
    'us-military-action-against-iran-in-2024',
    'iran-strike-on-israel-before-december',
    'another-iran-strike-on-israel-in-2024',
    'middle-east-regional-war-in-2024',
]

# Fetch market data from Gamma API
print('Fetching Iran-related Polymarket data...')
unique_markets = []
seen = set()

# Method 1: Tag-based search
for tag in ['geopolitics', 'politics', 'world', 'middle-east', 'military']:
    try:
        r = requests.get('https://gamma-api.polymarket.com/markets',
                        params={'tag': tag, 'limit': 200}, timeout=15)
        for m in r.json():
            if any(kw in m.get('question','').lower() or kw in m.get('description','').lower()
                   for kw in IRAN_KW):
                slug = m.get('slug', m.get('id', ''))
                if slug not in seen:
                    seen.add(slug)
                    unique_markets.append(m)
    except Exception:
        pass

# Method 2: Known slugs
for slug in known_slugs:
    if slug in seen:
        continue
    try:
        r = requests.get(f'https://gamma-api.polymarket.com/markets?slug={slug}', timeout=10)
        data = r.json()
        if data:
            unique_markets.append(data[0])
            seen.add(slug)
        time.sleep(0.15)
    except Exception:
        continue

print(f'Found {len(unique_markets)} Iran-related markets')

# Build enriched dataframe
enriched = []
for m in unique_markets:
    try:
        volume = float(m.get('volume', 0) or 0)
    except (ValueError, TypeError):
        volume = 0
    outcomes = m.get('outcomes', [])
    if isinstance(outcomes, str):
        try: outcomes = json.loads(outcomes)
        except: outcomes = []
    prices = m.get('outcomePrices', [])
    if isinstance(prices, str):
        try: prices = json.loads(prices)
        except: prices = []

    enriched.append({
        'question': m.get('question', ''),
        'slug': m.get('slug', ''),
        'market_id': m.get('id', ''),
        'volume': volume,
        'liquidity': float(m.get('liquidityNum', 0) or 0),
        'active': m.get('active', False),
        'closed': m.get('closed', False),
        'end_date': m.get('endDate', m.get('endDateIso', '')),
        'created_at': m.get('createdAt', ''),
        'outcomes': str(outcomes),
        'outcome_prices': str(prices),
        'description': str(m.get('description', ''))[:500],
        'clob_token_ids': m.get('clobTokenIds', ''),
    })

pm_df = pd.DataFrame(enriched).sort_values('volume', ascending=False).reset_index(drop=True)
pm_df.to_csv('../data/raw/polymarket_markets.csv', index=False)

print(f'\nSaved {len(pm_df)} markets to polymarket_markets.csv')
print(f'\nTop 10 by volume:')
for _, row in pm_df.head(10).iterrows():
    print(f'  ${row["volume"]:>12,.0f} | {row["question"][:55]}')

Fetching Iran-related Polymarket data...
Found 23 Iran-related markets

Saved 23 markets to polymarket_markets.csv

Top 10 by volume:
  $   4,992,689 | Will the Iranian regime fall in 2025?
  $   3,538,911 | Will Iran close the Strait of Hormuz in 2025?
  $   3,265,663 | Israel military response against Iran in October?
  $   3,168,666 | Another Iran strike on Israel in 2024?
  $   2,796,607 | US-Iran nuclear deal in 2025?
  $   1,725,722 | Will the US officially declare war on Iran in 2025?
  $   1,713,637 | Israel strikes Iran before 2026?
  $   1,523,164 | Iran Nuke in 2025?
  $   1,317,272 | Iran strike on Israel before December?
  $   1,185,191 | Another US military action against Iran before 2026?


## 5. Data Collection Summary

In [15]:
print('=== DATA COLLECTION SUMMARY ===')
print()
raw_dir = '../data/raw'
for f in sorted(os.listdir(raw_dir)):
    if f.endswith('.csv'):
        path = os.path.join(raw_dir, f)
        df = pd.read_csv(path)
        size_kb = os.path.getsize(path) / 1024
        print(f'{f:<30} {len(df):>6} rows  {size_kb:>8.1f} KB')

print()
print('Next step: Run notebook 02 to clean and merge these datasets.')

=== DATA COLLECTION SUMMARY ===

brent_daily.csv                   385 rows       6.4 KB
polymarket_markets.csv             23 rows      19.3 KB
truth_posts.csv                 21013 rows    6750.8 KB
uso_intraday.csv                 4557 rows     458.8 KB
wti_daily.csv                     385 rows       6.3 KB

Next step: Run notebook 02 to clean and merge these datasets.
